In [20]:
#Import packages
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import torch
import transformers as ppb
import warnings
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
import os
import matplotlib.pyplot as plt
import keras.ops as k_ops

In [21]:
#參數設置區

#資料集、模型選擇參數設置區
dataSet = 'Final_XSS_Dataset' #選項有 'XSS_dataset'、'HttpParamsDataset'、'PortSwigger_XSSDataSet'、'XSSed_CSIC2010'、'Final_XSS_Dataset'
modelType = 'BERT' #選項有 'BERT'、'DistilBERT'

#I/O參數設置區

if dataSet == 'XSS_dataset':
  dataSetPath = r'../res/TrainingData/DataSet/XSS_dataset.csv'
elif dataSet == 'HttpParamsDataset':
  dataSetPath = r'../res/TrainingData/DataSetOfHttpParamsDataset/payload_train.csv'
elif dataSet == 'PortSwigger_XSSDataSet':
  dataSetPath = r'../res/TrainingData/Final_PortSwigger_XSSDataset(資料集內容待驗證)/PortSwigger_XSSDataSet.csv'
elif dataSet == 'XSSed_CSIC2010':
  dataSetPath = r'../res/TrainingData/Final_XSSed_CSIC2010FromKaggle(資料集內容待驗證)/XSSed_CSIC2010.csv'
elif dataSet == 'Final_XSS_Dataset':
  dataSetPath = r'../res/TrainingData/Final_XSSDataSet(資料集內容待驗證)/Final_XSS_dataset(Train).csv'

In [22]:
#載入資料集
df = pd.read_csv( dataSetPath, header = None )

In [23]:
#Loading Pre-trained Model

#Loading Pre-trained BERT Model:
if modelType == 'DistilBERT':
  model_class, tokenizer_class, pretrained_weights = ( ppb.DistilBertModel, ppb.DistilBertTokenizer, 'distilbert-base-uncased' )
elif modelType == 'BERT':
  model_class, tokenizer_class, pretrained_weights = ( ppb.BertModel, ppb.BertTokenizer, 'bert-base-uncased' )

# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained( pretrained_weights )
model = model_class.from_pretrained( pretrained_weights )

In [24]:
#Debugging Cell (You can use this cell to debug if you encounter any problems in tokenization)

index = 0

for element in df[0]:
    if type( element ) != type( 'str' ):
        print( 'In index ', index, ': The type of element is ', type( element ) )

    index = index + 1

In [25]:
#Tokenization
# tokenized = df[0].apply( ( lambda x: tokenizer.encode( x, add_special_tokens = True ) ) ) #Stable and crediable method
tokenized = df[0].apply( ( lambda x: tokenizer.encode( x, add_special_tokens = True, truncation = True, max_length = 600 ) ) ) # New and unstable method 

In [26]:
#Padding
padded = []
labels = []
index = 0
maxLen = 0

for token in tokenized.values:

  if len( token ) > maxLen:
    maxLen = len( token )

for token in tokenized.values:
    
  while len( token ) <= maxLen:

    if len( token ) < maxLen:
      token.append( 0 )

    else:
      padded.append( token )

      if dataSet == 'HttpParamsDataset' :
        labels.append( df.loc[ index, 3 ] )

      else:
        labels.append( df.loc[ index, 1 ] )
          
      break

  index = index + 1

padded = np.array( padded )
print( type( padded ) )
print( 'The maxium length is ', maxLen )

<class 'numpy.ndarray'>
The maxium length is  600


In [27]:
#如果是使用HttpParamsDataset資料集，對資料集的label作一點處理
if dataSet == 'HttpParamsDataset' : 
  print( 'The length of labels is ', len(labels) ) #Debug用

  templabels = [] #將label從字串轉換成整數用，具體作法是先從label list中取出一個label，將其轉換成整數後存進此list中。
                  #當所有label都轉成整數並存進此list後會再將label list重新指向此list。

  for index in range( 0, len( labels ), 1 ):

    if labels[index] == 'norm':
      templabels.append( 0 )

    elif labels[index] == 'anom':
      templabels.append( 1 )

  print( 'The length of templabels is ', len(templabels) ) #Debug用
  labels = templabels

In [28]:
#Debug(印label)
print( labels )

[np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1)

In [29]:
#實作Transformer的Encoder(使用繼承方式建構神經層)
class TransformerEncoder( layers.Layer ):

    def __init__( self, embedDim, denseDim, numHeads, **kwargs  ): #不知為啥會多一個**kwargs
        super().__init__( **kwargs ) #別忘了要先呼叫父類別的初始化函式
                                     #不知為啥會多一個**kwargs

        #請在下方定義模型的神經層

        #STEP1:宣告並初始化與此神經層有關的變數
        self.embedDim = embedDim
        self.denseDim = denseDim
        self.numHeads = numHeads

        #STEP2:定義此神經層所含的元件的神經網路具體架構
        self.attention = layers.MultiHeadAttention (
                             num_heads = numHeads,
                             key_dim = embedDim
                         )

        self.layerNorm1 = layers.LayerNormalization()
        self.layerNorm2 = layers.LayerNormalization()

        self.dense1 = layers.Dense( denseDim, activation = "relu" )
        self.dense2 = layers.Dense( embedDim )

    # 在 call() 裡定義正向傳播的過程
    def call( self, inputs, mask = None ):

        #請在下方定義正向傳播的過程

        #STEP1:一些input資料必要的預處理

        #Embedding層生成的遮罩會是二維的，但 attention 層會期望接收三維的遮罩，
        #所以我們要增加其軸數(在第一軸增加一軸)
        if mask is not None:
            mask = mask[ :, tf.newaxis, : ]

        #STEP2:定義此神經層的正向傳播過程
        attentionOutput = self.attention( inputs, inputs, attention_mask = mask )
        projInput = self.layerNorm1( inputs + attentionOutput )
        projHiddenOutput1 = self.dense1( projInput )
        projOutput = self.dense2( projHiddenOutput1 )
        output = self.layerNorm2( projInput + projOutput )

        #STEP3:回傳此神經層的輸出值
        return output

    # 這個函式的功用請參考 [1] 的 p.11-47 ~ p.11-48
    def get_config( self ):
        config = super().get_config()
        config.update( {
                        "embedDim": self.embedDim,
                        "denseDim": self.denseDim,
                        "numHeads": self.numHeads,
                       } )
        return config

In [ ]:
#實作positional embedding(使用繼承方式建構神經層)
class PositionalEmbedding( layers.Layer ):

    def __init__( self, length, inputDim, outputDim, **kwargs ): #不知為啥會多一個**kwargs

        super().__init__( **kwargs )
        self.tokenEmbeddings = layers.Embedding( input_dim = inputDim, output_dim = outputDim ) #為token的索引準備一個Embedding layer
        self.positionEmbeddings = layers.Embedding( input_dim = length, output_dim = outputDim ) #為token的位置準備一個Embedding layer
        self.length = length
        self.inputDim = inputDim
        self.outputDim = outputDim

    def call( self, inputs ):
        positions = tf.range( start = 0, limit = tf.shape( inputs )[-1], delta = 1 )
        embeddedTokens = self.tokenEmbeddings( inputs )
        embeddedPositions = self.positionEmbeddings( positions )
        return embeddedTokens + embeddedPositions

    def compute_mask( self, inputs, mask = None ):
        return tf.math.not_equal( inputs, 0 )

    def get_config( self ):
        config = super().get_config()
        
        config.update( {
            "length": self.length,
            "inputDim": self.inputDim,
            "outputDim": self.outputDim,
        } )

        return config

In [33]:
#實作包含Transformer Encoder的模型

#STEP1: 設計此模型的架構
inputs = keras.Input( shape = ( None, ), dtype = "int64" )
wordEmbedded = PositionalEmbedding( length = maxLen, inputDim = 20000, outputDim = 768 )( inputs )
transformerOutputs = TransformerEncoder( embedDim = 768, denseDim = 32, numHeads = 2 )( wordEmbedded )  
#上面那行程式碼的相關參數說明
#  embedDim: 輸入token的向量大小
#  denseDim: Transformer Encoder 內部的 Fully Connection 裡某一個 layer 的大小。           
#  numHeads: Attention heads 的數量
decreaseDimensionalSpaceOutput = layers.GlobalMaxPooling1D()( transformerOutputs )
hiddenLayersOutputs = layers.Dense( 64, activation = "relu" )( decreaseDimensionalSpaceOutput )
outputs = layers.Dense( 2, activation = 'softmax' )( hiddenLayersOutputs )

#STEP2: 把模型給實例化(instantiated)
model = keras.Model( inputs = inputs, outputs = outputs )
model.summary()

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [ ]:
#載入模型(待確認這樣寫是否有成功載入權重)
modelPath = 'BestModel_Transformer.keras'

model = keras.models.load_model (
    modelPath,
    custom_objects = { "TransformerEncoder": TransformerEncoder,
                       "PositionalEmbedding": PositionalEmbedding }
)

In [ ]:
#處理測試資料
data = padded 
labels = to_categorical( labels )
labels = np.array( labels )

#查看結果
print( 'Testing data and label : ', data.shape, labels.shape )

Testing data and label :  (3307, 600) (3307, 2)


In [ ]:
#訓練參數設置(此配置可能跟論文的配置不同)
batchSize = 64

In [ ]:
#開始測試
history = model.evaluate( data, labels, batch_size = batchSize )

52/52 [==============================] - 3s 54ms/step - loss: 0.0060 - accuracy: 0.9976 - recall: 0.9976
